# Etapa 4 - Modelagem com Machine Learning

In [ ]:
#modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC 
#embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
#treino
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
#metricas
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
#manipulacao
import pandas as pd
import numpy as np
#visualizacao
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

In [3]:
PATH_PARQUET = '../data/dataset_tidy.parquet'   # ajuste o caminho
PATH_ACCORD = '../data/accord_official_features.csv'   # ajuste o caminho
PATH_BASE = '../data/base_construida_features.csv'   # ajuste o caminho      

In [4]:
df_parquet = pd.read_parquet(PATH_PARQUET)
df_accord = pd.read_csv(PATH_ACCORD)
df_base = pd.read_csv(PATH_BASE)
print('Bases de dados carregadas:')
print(f' - Parquet: {len(df_parquet)} amostras')
print(f' - ACCORD Official: {len(df_accord)} amostras')
print(f' - Base Construída: {len(df_base)} amostras')

Bases de dados carregadas:
 - Parquet: 2942 amostras
 - ACCORD Official: 2334 amostras
 - Base Construída: 2768 amostras


In [5]:
print("Parquet: regras vs. não-regras\n", df_parquet['is_rule'].value_counts())
print("ACCORD Official: regras vs. não-regras\n", df_accord['is_rule'].value_counts())
print("Base Construída: regras vs. não-regras\n", df_base['is_rule'].value_counts())


Parquet: regras vs. não-regras
 is_rule
0    1708
1    1234
Name: count, dtype: int64
ACCORD Official: regras vs. não-regras
 is_rule
0    1566
1     768
Name: count, dtype: int64
Base Construída: regras vs. não-regras
 is_rule
0.0    1588
1.0    1180
Name: count, dtype: int64


In [6]:
def train_models(representation, base_name, X_train_text, X_test_text,y_train, y_test):
    resultados = []

    # ── Regressão Logística ───────────────────────────────────────────────────
    lr = LogisticRegression(class_weight='balanced', max_iter=1000,
                             random_state=42, C=1.0)
    lr.fit(X_train_text, y_train)
    y_pred_lr = lr.predict(X_test_text)
    print('═'*55)
    print(f'REGRESSÃO LOGÍSTICA — {representation} ({base_name})')
    print('═'*55)
    print(classification_report(y_test, y_pred_lr, target_names=['Não-Regra','Regra']))
    resultados.append(('Logistic Regression', lr, X_test_text, y_pred_lr))

    # ── Random Forest ─────────────────────────────────────────────────────────
    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample',
                                 random_state=42, n_jobs=-1)
    rf.fit(X_train_text, y_train)
    y_pred_rf = rf.predict(X_test_text)
    print('═'*55)
    print(f'RANDOM FOREST — {representation} ({base_name})')
    print('═'*55)
    print(classification_report(y_test, y_pred_rf, target_names=['Não-Regra','Regra']))
    resultados.append(('Random Forest', rf, X_test_text, y_pred_rf))

    # ── SVM ───────────────────────────────────────────────────────────────────
    svm = SVC(kernel='linear', class_weight='balanced', probability=True,
               random_state=42, C=1.0)
    svm.fit(X_train_text, y_train)
    y_pred_svm = svm.predict(X_test_text)
    print('═'*55)
    print(f'SVM — {representation} ({base_name})')
    print('═'*55)
    print(classification_report(y_test, y_pred_svm, target_names=['Não-Regra','Regra']))
    resultados.append(('SVM', svm, X_test_text, y_pred_svm))

    return resultados, [y_pred_lr, y_pred_rf, y_pred_svm]

## Avaliando desempenho PARQUET

In [7]:
x_text_p = df_parquet['Text']
y_p      = df_parquet['is_rule'].values
X_train_text_p, X_test_text_p, y_train_p, y_test_p = train_test_split(x_text_p, y_p, test_size=0.2, random_state=42)

print(f'Treino : {len(y_train_p)} amostras '
      f'({y_train_p.sum()} regras / {(y_train_p==0).sum()} não-regras)')
print(f'Teste  : {len(y_test_p)} amostras '
      f'({y_test_p.sum()} regras / {(y_test_p==0).sum()} não-regras)')
print()
print(f'% regras no treino : {y_train_p.mean()*100:.1f}%')
print(f'% regras no teste  : {y_test_p.mean()*100:.1f}%')     


Treino : 2353 amostras (985 regras / 1368 não-regras)
Teste  : 589 amostras (249 regras / 340 não-regras)

% regras no treino : 41.9%
% regras no teste  : 42.3%


In [8]:
# ── Vetorização TF-IDF ───────────────────────────────────────────────────────
# ngram_range=(1,2): unigramas + bigramas captura padrões como "shall not", "must be"
# max_features=5000: top 5000 n-gramas por TF-IDF
# sublinear_tf=True: aplica log na frequência — reduz peso de termos muito frequentes
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    stop_words=None    # mantém 'not', 'shall', etc. — são informativos aqui
)

X_train_tfidf_p = tfidf.fit_transform(X_train_text_p)
X_test_tfidf_p  = tfidf.transform(X_test_text_p)

print(f'Shape TF-IDF treino: {X_train_tfidf_p.shape}')
print(f'Shape TF-IDF teste : {X_test_tfidf_p.shape}')

Shape TF-IDF treino: (2353, 5000)
Shape TF-IDF teste : (589, 5000)


In [9]:
list_y_pred_p = []
results_parquet, list_y_pred_p = train_models('TF-IDF', 'Parquet', X_train_tfidf_p, X_test_tfidf_p, y_train_p, y_test_p)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — TF-IDF (Parquet)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.90      0.87      0.89       340
       Regra       0.83      0.86      0.85       249

    accuracy                           0.87       589
   macro avg       0.87      0.87      0.87       589
weighted avg       0.87      0.87      0.87       589

═══════════════════════════════════════════════════════
RANDOM FOREST — TF-IDF (Parquet)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.90      0.86      0.88       340
       Regra       0.82      0.87      0.85       249

    accuracy                           0.87       589
   macro avg       0.86      0.87      0.86       589
weighted avg       0.87      0.87      0.87       589

══════════════════════════════════════════════════

## Avaliando desempenho ACCORD 

In [10]:
x_text_a = df_accord['Text']
y_a      = df_accord['is_rule'].values
X_train_text_a, X_test_text_a, y_train_a, y_test_a = train_test_split(x_text_a, y_a, test_size=0.2, random_state=42)

print(f'Treino : {len(y_train_a)} amostras '
      f'({y_train_a.sum()} regras / {(y_train_a==0).sum()} não-regras)')
print(f'Teste  : {len(y_test_a)} amostras '
      f'({y_test_a.sum()} regras / {(y_test_a==0).sum()} não-regras)')
print()
print(f'% regras no treino : {y_train_a.mean()*100:.1f}%')
print(f'% regras no teste  : {y_test_a.mean()*100:.1f}%')     


Treino : 1867 amostras (606 regras / 1261 não-regras)
Teste  : 467 amostras (162 regras / 305 não-regras)

% regras no treino : 32.5%
% regras no teste  : 34.7%


In [11]:
tfidf_a = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    stop_words=None    # mantém 'not', 'shall', etc. — são informativos aqui
)

X_train_tfidf_a = tfidf_a.fit_transform(X_train_text_a)
X_test_tfidf_a  = tfidf_a.transform(X_test_text_a)

print(f'Shape TF-IDF treino: {X_train_tfidf_a.shape}')
print(f'Shape TF-IDF teste : {X_test_tfidf_a.shape}')


Shape TF-IDF treino: (1867, 5000)
Shape TF-IDF teste : (467, 5000)


In [12]:
list_y_pred_a = []
results_accord, list_y_pred_a = train_models('TF-IDF', 'ACCORD Official', X_train_tfidf_a, X_test_tfidf_a, y_train_a, y_test_a)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — TF-IDF (ACCORD Official)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.93      0.90      0.91       305
       Regra       0.82      0.87      0.84       162

    accuracy                           0.89       467
   macro avg       0.87      0.88      0.88       467
weighted avg       0.89      0.89      0.89       467

═══════════════════════════════════════════════════════
RANDOM FOREST — TF-IDF (ACCORD Official)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.90      0.93      0.92       305
       Regra       0.87      0.81      0.84       162

    accuracy                           0.89       467
   macro avg       0.88      0.87      0.88       467
weighted avg       0.89      0.89      0.89       467

══════════════════════════════════

## Avaliando desempenho base desenvolvida

In [13]:
x_text_b = df_base['Text']
y_b      = df_base['is_rule'].values
X_train_text_b, X_test_text_b, y_train_b, y_test_b = train_test_split(x_text_b, y_b, test_size=0.2, random_state=42)

print(f'Treino : {len(y_train_b)} amostras '
      f'({y_train_b.sum()} regras / {(y_train_b==0).sum()} não-regras)')
print(f'Teste  : {len(y_test_b)} amostras '
      f'({y_test_b.sum()} regras / {(y_test_b==0).sum()} não-regras)')
print()
print(f'% regras no treino : {y_train_b.mean()*100:.1f}%')
print(f'% regras no teste  : {y_test_b.mean()*100:.1f}%')     


Treino : 2214 amostras (934.0 regras / 1280 não-regras)
Teste  : 554 amostras (246.0 regras / 308 não-regras)

% regras no treino : 42.2%
% regras no teste  : 44.4%


In [14]:
tfidf_b = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    stop_words=None
)

X_train_tfidf_b = tfidf_b.fit_transform(X_train_text_b)
X_test_tfidf_b  = tfidf_b.transform(X_test_text_b)

print(f'Shape TF-IDF treino: {X_train_tfidf_b.shape}')
print(f'Shape TF-IDF teste : {X_test_tfidf_b.shape}')


Shape TF-IDF treino: (2214, 5000)
Shape TF-IDF teste : (554, 5000)


In [15]:
list_y_pred_b = []
results_base, list_y_pred_b = train_models('TF-IDF', 'Base Construída', X_train_tfidf_b, X_test_tfidf_b, y_train_b, y_test_b)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — TF-IDF (Base Construída)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.90      0.85      0.87       308
       Regra       0.82      0.88      0.85       246

    accuracy                           0.86       554
   macro avg       0.86      0.86      0.86       554
weighted avg       0.87      0.86      0.86       554

═══════════════════════════════════════════════════════
RANDOM FOREST — TF-IDF (Base Construída)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.91      0.85      0.88       308
       Regra       0.82      0.90      0.86       246

    accuracy                           0.87       554
   macro avg       0.87      0.87      0.87       554
weighted avg       0.87      0.87      0.87       554

══════════════════════════════════

### Resumo do baseline

In [16]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

def summarize(nome_modelo, nome_base, representacao, y_true, y_pred):
    return {
        'Base':        nome_base,
        'Modelo':      nome_modelo,
        'Representação': representacao,
        'Accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'Precision':   round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'Recall':      round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'F1-Regra':    round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'F1-NRegra':   round(f1_score(y_true, y_pred, pos_label=0, zero_division=0), 4),
        'F1-Macro':    round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'F1-Weighted': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }

In [17]:
resultados = [
    # ── PARQUET ──────────────────────────────────────────────────────────────
    summarize('Logistic Regression', 'Parquet', 'TF-IDF', y_test_p, list_y_pred_p[0]),
    summarize('Random Forest','Parquet', 'TF-IDF', y_test_p, list_y_pred_p[1]),
    summarize('SVM','Parquet', 'TF-IDF', y_test_p, list_y_pred_p[2]),
    # ── ACCORD OFFICIAL ──────────────────────────────────────────────────────
    summarize('Logistic Regression', 'ACCORD Official', 'TF-IDF', y_test_a, list_y_pred_a[0]),
    summarize('Random Forest','ACCORD Official', 'TF-IDF', y_test_a, list_y_pred_a[1]),
    summarize('SVM','ACCORD Official', 'TF-IDF', y_test_a, list_y_pred_a[2]),
    # ── BASE CONSTRUÍDA ───────────────────────────────────────────────────────
    summarize('Logistic Regression', 'Base Construída', 'TF-IDF', y_test_b, list_y_pred_b[0]),
    summarize('Random Forest','Base Construída', 'TF-IDF', y_test_b, list_y_pred_b[1]),
    summarize('SVM','Base Construída', 'TF-IDF', y_test_b, list_y_pred_b[2]),
]

df_summary = pd.DataFrame(resultados)

print('SUMMARY — BASELINE TF-IDF')
print(df_summary.to_string(index=False))


SUMMARY — BASELINE TF-IDF
           Base              Modelo Representação  Accuracy  Precision  Recall  F1-Regra  F1-NRegra  F1-Macro  F1-Weighted
        Parquet Logistic Regression        TF-IDF    0.8693     0.8333  0.8635    0.8481     0.8852    0.8667       0.8696
        Parquet       Random Forest        TF-IDF    0.8659     0.8244  0.8675    0.8454     0.8816    0.8635       0.8663
        Parquet                 SVM        TF-IDF    0.8862     0.8447  0.8956    0.8694     0.8992    0.8843       0.8866
ACCORD Official Logistic Regression        TF-IDF    0.8865     0.8150  0.8704    0.8418     0.9115    0.8767       0.8873
ACCORD Official       Random Forest        TF-IDF    0.8908     0.8675  0.8086    0.8371     0.9179    0.8775       0.8898
ACCORD Official                 SVM        TF-IDF    0.8715     0.7931  0.8519    0.8214     0.8997    0.8605       0.8725
Base Construída Logistic Regression        TF-IDF    0.8628     0.8220  0.8821    0.8510     0.8729    0.8619    

In [18]:
# ── Destaca o melhor por base ─────────────────────────────────────────────────
print('\nMelhor modelo por base (F1-Macro):')
for base in df_summary['Base'].unique():
    sub = df_summary[df_summary['Base'] == base]
    best = sub.loc[sub['F1-Macro'].idxmax()]
    print(f'   {base:<20} → {best["Modelo"]:<22} F1-Macro={best["F1-Macro"]:.4f}')


Melhor modelo por base (F1-Macro):
   Parquet              → SVM                    F1-Macro=0.8843
   ACCORD Official      → Random Forest          F1-Macro=0.8775
   Base Construída      → SVM                    F1-Macro=0.8711


In [19]:
df_summary.to_csv('../data/summary_baseline_tfidf.csv', index=False)

## Utilizando mais features + TFIDF

In [20]:
FEAT_COLS = ['n_deontic', 'has_shall', 'has_should']

### Parquet

In [21]:
# Extrai features pelos índices do split
feat_train = df_parquet.loc[X_train_text_p.index, FEAT_COLS].values.astype(float)
feat_test  = df_parquet.loc[X_test_text_p.index,  FEAT_COLS].values.astype(float)

# Normaliza
scaler     = StandardScaler()
feat_train = scaler.fit_transform(feat_train)
feat_test  = scaler.transform(feat_test)

# Concatena TF-IDF + features
X_train_comb_p = hstack([X_train_tfidf_p, csr_matrix(feat_train)])
X_test_comb_p  = hstack([X_test_tfidf_p,  csr_matrix(feat_test)])

In [23]:
list_y_pred_p_feature = []
res_p, list_y_pred_p_feature = train_models('TF-IDF + Feature', 'Parquet', X_train_comb_p, X_test_comb_p, y_train_p, y_test_p)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — TF-IDF + Feature (Parquet)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.88      0.83      0.86       340
       Regra       0.79      0.85      0.82       249

    accuracy                           0.84       589
   macro avg       0.83      0.84      0.84       589
weighted avg       0.84      0.84      0.84       589

═══════════════════════════════════════════════════════
RANDOM FOREST — TF-IDF + Feature (Parquet)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.91      0.88      0.90       340
       Regra       0.84      0.88      0.86       249

    accuracy                           0.88       589
   macro avg       0.88      0.88      0.88       589
weighted avg       0.88      0.88      0.88       589

══════════════════════════════

### ACCORD

In [26]:
# Extrai features pelos índices do split
feat_train = df_accord.loc[X_train_text_a.index, FEAT_COLS].values.astype(float)
feat_test  = df_accord.loc[X_test_text_a.index,  FEAT_COLS].values.astype(float)

# Normaliza
scaler     = StandardScaler()
feat_train = scaler.fit_transform(feat_train)
feat_test  = scaler.transform(feat_test)

# Concatena TF-IDF + features
X_train_comb_a = hstack([X_train_tfidf_a, csr_matrix(feat_train)])
X_test_comb_a  = hstack([X_test_tfidf_a,  csr_matrix(feat_test)])

In [27]:
list_y_pred_a_feature = []
res_a, list_y_pred_a_feature = train_models('TF-IDF + Feature', 'ACCORD Official', X_train_comb_a, X_test_comb_a, y_train_a, y_test_a)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — TF-IDF + Feature (ACCORD Official)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.91      0.85      0.88       305
       Regra       0.75      0.85      0.79       162

    accuracy                           0.85       467
   macro avg       0.83      0.85      0.84       467
weighted avg       0.86      0.85      0.85       467

═══════════════════════════════════════════════════════
RANDOM FOREST — TF-IDF + Feature (ACCORD Official)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.90      0.93      0.92       305
       Regra       0.86      0.81      0.84       162

    accuracy                           0.89       467
   macro avg       0.88      0.87      0.88       467
weighted avg       0.89      0.89      0.89       467

══════════════

### Base construida

In [28]:
# Extrai features pelos índices do split
feat_train = df_base.loc[X_train_text_b.index, FEAT_COLS].values.astype(float)
feat_test  = df_base.loc[X_test_text_b.index,  FEAT_COLS].values.astype(float)

# Normaliza
scaler     = StandardScaler()
feat_train = scaler.fit_transform(feat_train)
feat_test  = scaler.transform(feat_test)

# Concatena TF-IDF + features
X_train_comb_b = hstack([X_train_tfidf_b, csr_matrix(feat_train)])
X_test_comb_b  = hstack([X_test_tfidf_b,  csr_matrix(feat_test)])

In [29]:
list_y_pred_b_feature = []
res_b, list_y_pred_b_feature = train_models('TF-IDF + Feature', 'Base Construída', X_train_comb_b, X_test_comb_b, y_train_b, y_test_b)

═══════════════════════════════════════════════════════
REGRESSÃO LOGÍSTICA — TF-IDF + Feature (Base Construída)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.88      0.84      0.86       308
       Regra       0.81      0.86      0.83       246

    accuracy                           0.85       554
   macro avg       0.85      0.85      0.85       554
weighted avg       0.85      0.85      0.85       554

═══════════════════════════════════════════════════════
RANDOM FOREST — TF-IDF + Feature (Base Construída)
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

   Não-Regra       0.91      0.87      0.89       308
       Regra       0.85      0.90      0.87       246

    accuracy                           0.88       554
   macro avg       0.88      0.88      0.88       554
weighted avg       0.88      0.88      0.88       554

══════════════

### Resumo 

In [30]:
# Mapeia resultados por base
configs = [
    (res_p, 'Parquet',          y_test_p),
    (res_a, 'ACCORD Official',  y_test_a),
    (res_b, 'Base Construída',  y_test_b),
]

# nome_modelo, nome_base, representacao, y_true, y_pred)

resultados_feat = []
for res, nome_base, y_test in configs:
    for nome_modelo, model, X_test_comb, y_pred in res:
        resultados_feat.append(
            summarize(nome_modelo, nome_base, 'TF-IDF + Features', y_test, y_pred)
        )

df_summary_feat = pd.DataFrame(resultados_feat)

print('╔══ SUMMARY — TF-IDF + FEATURES (3 modelos × 3 bases) ════════════════════╗')
print(df_summary_feat.to_string(index=False))
print('╚══════════════════════════════════════════════════════════════════════════╝')

print('\n🏆 Melhor modelo por base (F1-Macro):')
for base in df_summary_feat['Base'].unique():
    sub  = df_summary_feat[df_summary_feat['Base'] == base]
    best = sub.loc[sub['F1-Macro'].idxmax()]
    print(f'   {base:<20} → {best["Modelo"]:<22} F1-Macro={best["F1-Macro"]:.4f}')

df_summary_feat.to_csv('../data/summary_tfidf_features.csv', index=False)

╔══ SUMMARY — TF-IDF + FEATURES (3 modelos × 3 bases) ════════════════════╗
           Base              Modelo     Representação  Accuracy  Precision  Recall  F1-Regra  F1-NRegra  F1-Macro  F1-Weighted
        Parquet Logistic Regression TF-IDF + Features    0.8387     0.7873  0.8474    0.8162     0.8563    0.8363       0.8394
        Parquet       Random Forest TF-IDF + Features    0.8812     0.8429  0.8835    0.8627     0.8952    0.8790       0.8815
        Parquet                 SVM TF-IDF + Features    0.8829     0.8409  0.8916    0.8655     0.8962    0.8809       0.8832
ACCORD Official Logistic Regression TF-IDF + Features    0.8480     0.7486  0.8457    0.7942     0.8795    0.8368       0.8499
ACCORD Official       Random Forest TF-IDF + Features    0.8887     0.8571  0.8148    0.8354     0.9159    0.8757       0.8880
ACCORD Official                 SVM TF-IDF + Features    0.8779     0.8144  0.8395    0.8267     0.9058    0.8663       0.8784
Base Construída Logistic Regression

## Comparação Baseline com features

In [31]:
# ── Comparação direta: TF-IDF puro vs TF-IDF + Features ──────────────────────
df_comparacao = pd.concat([df_summary, df_summary_feat], ignore_index=True)

print('╔══ COMPARAÇÃO: TF-IDF vs TF-IDF + Features ══════════════════════════════╗')
pivot = df_comparacao.pivot_table(
    index=['Base', 'Modelo'],
    columns='Representação',
    values='F1-Macro'
).round(4)

pivot['Δ F1-Macro'] = (pivot['TF-IDF + Features'] - pivot['TF-IDF']).round(4)
pivot['Melhora?'] = pivot['Δ F1-Macro'].apply(
    lambda x: 'SIM' if x > 0.005 else ('NEUTRO' if abs(x) <= 0.005 else 'PIORA')
)
print(pivot.to_string())
print('╚══════════════════════════════════════════════════════════════════════════╝')

df_comparacao.to_csv('../data/summary_completo_baseline.csv', index=False)

╔══ COMPARAÇÃO: TF-IDF vs TF-IDF + Features ══════════════════════════════╗
Representação                        TF-IDF  TF-IDF + Features  Δ F1-Macro Melhora?
Base            Modelo                                                             
ACCORD Official Logistic Regression  0.8767             0.8368     -0.0399    PIORA
                Random Forest        0.8775             0.8757     -0.0018   NEUTRO
                SVM                  0.8605             0.8663      0.0058      SIM
Base Construída Logistic Regression  0.8619             0.8472     -0.0147    PIORA
                Random Forest        0.8694             0.8818      0.0124      SIM
                SVM                  0.8711             0.8727      0.0016   NEUTRO
Parquet         Logistic Regression  0.8667             0.8363     -0.0304    PIORA
                Random Forest        0.8635             0.8790      0.0155      SIM
                SVM                  0.8843             0.8809     -0.0034   NEUTRO


## Utilizando Validação cruzada

In [33]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

SCORING = {
    'f1_macro':    'f1_macro',
    'f1_weighted': 'f1_weighted',
    'accuracy':    'accuracy',
}

In [34]:
def cv_tfidf(df, nome_base, modelos):   
    X_text = df['Text'].values
    y      = df['is_rule'].values

    resultados = []
    print(f'\n{"="*60}')
    print(f'CV — TF-IDF puro | Base: {nome_base}')
    print(f'{"="*60}')

    for nome_modelo, modelo in modelos.items():
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000,
                                       sublinear_tf=True)),
            ('clf',   modelo)
        ])

        scores = cross_validate(pipe, X_text, y,
                                cv=cv, scoring=SCORING,
                                n_jobs=-1, return_train_score=False)

        resultados.append({
            'Base':          nome_base,
            'Representação': 'TF-IDF',
            'Modelo':        nome_modelo,
            'F1-Macro':      round(scores['test_f1_macro'].mean(), 4),
            'F1-Macro Std':  round(scores['test_f1_macro'].std(),  4),
            'F1-Weighted':   round(scores['test_f1_weighted'].mean(), 4),
            'Accuracy':      round(scores['test_accuracy'].mean(), 4),
        })

        print(f'  {nome_modelo:<25} '
              f'F1-Macro={scores["test_f1_macro"].mean():.4f} '
              f'± {scores["test_f1_macro"].std():.4f}')

    return resultados

In [35]:
from sklearn.base import BaseEstimator, TransformerMixin

class TfidfPlusFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, feat_cols, max_features=5000):
        self.feat_cols    = feat_cols
        self.max_features = max_features
        self.tfidf        = TfidfVectorizer(ngram_range=(1,2),
                                             max_features=max_features,
                                             sublinear_tf=True)
        self.scaler       = StandardScaler()

    def fit(self, X, y=None):
        # X é o DataFrame completo com Text + features
        self.tfidf.fit(X['Text'].fillna(''))
        self.scaler.fit(X[self.feat_cols].values.astype(float))
        return self

    def transform(self, X):
        tfidf_matrix = self.tfidf.transform(X['Text'].fillna(''))
        feat_matrix  = self.scaler.transform(
            X[self.feat_cols].values.astype(float)
        )
        return hstack([tfidf_matrix, csr_matrix(feat_matrix)])


def cv_tfidf_features(df, base_name, models):

    X = df[['Text'] + FEAT_COLS].copy()
    X['Text'] = X['Text'].fillna('')
    y = df['is_rule'].values

    resultados = []
    print(f'\n{"="*60}')
    print(f'CV — TF-IDF + Features | Base: {base_name}')
    print(f'{"="*60}')

    for nome_modelo, modelo in models.items():
        pipe = Pipeline([
            ('repr', TfidfPlusFeatures(feat_cols=FEAT_COLS)),
            ('clf',  modelo)
        ])

        scores = cross_validate(pipe, X, y,
                                cv=cv, scoring=SCORING,
                                n_jobs=-1, return_train_score=False)

        resultados.append({
            'Base':          base_name,
            'Representação': 'TF-IDF + Features',
            'Modelo':        nome_modelo,
            'F1-Macro':      round(scores['test_f1_macro'].mean(), 4),
            'F1-Macro Std':  round(scores['test_f1_macro'].std(),  4),
            'F1-Weighted':   round(scores['test_f1_weighted'].mean(), 4),
            'Accuracy':      round(scores['test_accuracy'].mean(), 4),
        })

        print(f'  {nome_modelo:<25} '
              f'F1-Macro={scores["test_f1_macro"].mean():.4f} '
              f'± {scores["test_f1_macro"].std():.4f}')

    return resultados

In [ ]:
modelos = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample',random_state=42, n_jobs=-1),
    'SVM': SVC(kernel='linear', class_weight='balanced', probability=True, random_state=42),
}


In [37]:
# TFIDF
all_results_tfidf = []
for df_b, nome in [(df_parquet, 'Parquet'),
                    (df_accord,  'ACCORD Official'),
                    (df_base,    'Base Construída')]:
    all_results_tfidf += cv_tfidf(df_b, nome, modelos)


CV — TF-IDF puro | Base: Parquet
  Logistic Regression       F1-Macro=0.8613 ± 0.0087
  Random Forest             F1-Macro=0.8850 ± 0.0109
  SVM                       F1-Macro=0.8671 ± 0.0131

CV — TF-IDF puro | Base: ACCORD Official
  Logistic Regression       F1-Macro=0.8269 ± 0.0167
  Random Forest             F1-Macro=0.8428 ± 0.0109
  SVM                       F1-Macro=0.8359 ± 0.0236

CV — TF-IDF puro | Base: Base Construída
  Logistic Regression       F1-Macro=0.8652 ± 0.0149
  Random Forest             F1-Macro=0.8795 ± 0.0104
  SVM                       F1-Macro=0.8711 ± 0.0155


In [38]:
#  TF-IDF + Features 
all_results_feat = []
for df_b, nome in [(df_parquet, 'Parquet'),
                    (df_accord,  'ACCORD Official'),
                    (df_base,    'Base Construída')]:
    all_results_feat += cv_tfidf_features(df_b, nome, modelos)


CV — TF-IDF + Features | Base: Parquet
  Logistic Regression       F1-Macro=0.8507 ± 0.0088
  Random Forest             F1-Macro=0.8874 ± 0.0105
  SVM                       F1-Macro=0.8741 ± 0.0154

CV — TF-IDF + Features | Base: ACCORD Official
  Logistic Regression       F1-Macro=0.7983 ± 0.0210
  Random Forest             F1-Macro=0.8520 ± 0.0074
  SVM                       F1-Macro=0.8374 ± 0.0208

CV — TF-IDF + Features | Base: Base Construída
  Logistic Regression       F1-Macro=0.8506 ± 0.0163
  Random Forest             F1-Macro=0.8829 ± 0.0148
  SVM                       F1-Macro=0.8678 ± 0.0173


In [42]:
df_cv = pd.DataFrame(all_results_tfidf + all_results_feat)
df_cv


,Base,Representação,Modelo,F1-Macro,F1-Macro Std,F1-Weighted,Accuracy
0,Parquet,TF-IDF,Logistic Regression,0.8613,0.0087,0.8642,0.8637
1,Parquet,TF-IDF,Random Forest,0.8850,0.0109,0.8877,0.8875
2,Parquet,TF-IDF,SVM,0.8671,0.0131,0.8700,0.8695
3,ACCORD Official,TF-IDF,Logistic Regression,0.8269,0.0167,0.8438,0.8410
4,ACCORD Official,TF-IDF,Random Forest,0.8428,0.0109,0.8622,0.8633
5,ACCORD Official,TF-IDF,SVM,0.8359,0.0236,0.8518,0.8492
6,Base Construída,TF-IDF,Logistic Regression,0.8652,0.0149,0.8675,0.8670
7,Base Construída,TF-IDF,Random Forest,0.8795,0.0104,0.8818,0.8815
8,Base Construída,TF-IDF,SVM,0.8711,0.0155,0.8733,0.8728
9,Parquet,TF-IDF + Features,Logistic Regression,0.8507,0.0088,0.8538,0.8532


In [43]:
print('\n╔══ IMPACTO DAS FEATURES LINGUÍSTICAS (Δ F1-Macro) ═══════════════════════╗')
pivot_cv = df_cv.pivot_table(
    index=['Base', 'Modelo'],
    columns='Representação',
    values='F1-Macro'
).round(4)

if 'TF-IDF + Features' in pivot_cv.columns and 'TF-IDF' in pivot_cv.columns:
    pivot_cv['Δ F1-Macro'] = (pivot_cv['TF-IDF + Features'] - pivot_cv['TF-IDF']).round(4)
    pivot_cv['Resultado'] = pivot_cv['Δ F1-Macro'].apply(
        lambda x: 'MELHORA' if x > 0.005
                  else ('NEUTRO' if abs(x) <= 0.005
                  else 'PIORA')
    )
print(pivot_cv.to_string())
print('╚══════════════════════════════════════════════════════════════════════════╝')


╔══ IMPACTO DAS FEATURES LINGUÍSTICAS (Δ F1-Macro) ═══════════════════════╗
Representação                        TF-IDF  TF-IDF + Features  Δ F1-Macro Resultado
Base            Modelo                                                              
ACCORD Official Logistic Regression  0.8269             0.7983     -0.0286     PIORA
                Random Forest        0.8428             0.8520      0.0092   MELHORA
                SVM                  0.8359             0.8374      0.0015    NEUTRO
Base Construída Logistic Regression  0.8652             0.8506     -0.0146     PIORA
                Random Forest        0.8795             0.8829      0.0034    NEUTRO
                SVM                  0.8711             0.8678     -0.0033    NEUTRO
Parquet         Logistic Regression  0.8613             0.8507     -0.0106     PIORA
                Random Forest        0.8850             0.8874      0.0024    NEUTRO
                SVM                  0.8671             0.8741      0.007

In [44]:
df_cv.to_csv('../data/summary_cv_5fold.csv', index=False)

## Avaliar embeddings do Word2vec

In [48]:
import re
def tokenize(text):
    text = re.sub(r'[^\w\s]', ' ', str(text).lower())
    return [t for t in text.split() if len(t) > 1]

In [49]:
def train_w2v(df, nome_base):
    sentences = [tokenize(t) for t in df['Text'].fillna('')]
    model = Word2Vec(
        sentences=sentences,
        vector_size=300,
        window=5,
        min_count=2,
        workers=4,
        seed=42,
        epochs=10
    )
    print(f'Word2Vec ({nome_base}): vocabulário={len(model.wv)} palavras')
    return model

In [50]:
w2v_p = train_w2v(df_parquet,'Parquet')
w2v_a = train_w2v(df_accord,'ACCORD Official')
w2v_b = train_w2v(df_base,'Base Construída')

Word2Vec (Parquet): vocabulário=3167 palavras
Word2Vec (ACCORD Official): vocabulário=2674 palavras
Word2Vec (Base Construída): vocabulário=2961 palavras


In [51]:
# Mean pooling sobre os vetores Word2Vec — padrão para sentence embeddings estáticos
def sentence_embedding_w2v(text, model, vector_size=300):
    tokens = tokenize(text)
    vectors = [
        model.wv[token]
        for token in tokens
        if token in model.wv
    ]
    if len(vectors) == 0:
        return np.zeros(vector_size)   # sentença sem tokens conhecidos
    return np.mean(vectors, axis=0)

def get_w2v_embeddings(df, model, vector_size=300):
    embeddings = np.vstack([
        sentence_embedding_w2v(text, model, vector_size)
        for text in df['Text'].fillna('')
    ])
    return embeddings

In [52]:
#gerar embeddings
emb_w2v_p = get_w2v_embeddings(df_parquet, w2v_p)
emb_w2v_a = get_w2v_embeddings(df_accord,  w2v_a)
emb_w2v_b = get_w2v_embeddings(df_base,    w2v_b)
print(f'Shape Parquet        : {emb_w2v_p.shape}')   
print(f'Shape ACCORD Official: {emb_w2v_a.shape}')
print(f'Shape Base Construída: {emb_w2v_b.shape}')

Shape Parquet        : (2942, 300)
Shape ACCORD Official: (2334, 300)
Shape Base Construída: (2768, 300)


In [53]:
from sklearn.preprocessing import normalize
def train_embed_w2v(nome_base, emb_full, df, modelos):
    y = df['is_rule'].values

    # Split estratificado
    X_train_emb, X_test_emb, y_train, y_test = train_test_split(
        emb_full, y, test_size=0.2, random_state=42, stratify=y
    )

    # Normalização L2 — importante para Word2Vec com SVM e LR
    X_train_emb = normalize(X_train_emb, norm='l2')
    X_test_emb  = normalize(X_test_emb,  norm='l2')

    resultados = []
    print(f'\n{"="*60}')
    print(f'Word2Vec | Base: {nome_base}')
    print(f'{"="*60}')

    for nome_modelo, modelo in modelos.items():
        modelo.fit(X_train_emb, y_train)
        y_pred = modelo.predict(X_test_emb)

        print(f'\n── {nome_modelo} ──')
        print(classification_report(y_test, y_pred,
                                    target_names=['Não-Regra','Regra']))

        resultados.append({
            'Base':          nome_base,
            'Representação': 'Word2Vec',
            'Modelo':        nome_modelo,
            'Accuracy':      round(accuracy_score(y_test, y_pred), 4),
            'Precision':     round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'Recall':        round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'F1-Regra':      round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
            'F1-NRegra':     round(f1_score(y_test, y_pred, pos_label=0, zero_division=0), 4),
            'F1-Macro':      round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4),
            'F1-Weighted':   round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        })

    return resultados

In [54]:
modelos_w2v = {
    'Logistic Regression': LogisticRegression(class_weight='balanced',max_iter=1000, random_state=42),
    'Random Forest':RandomForestClassifier(n_estimators=200,class_weight='balanced_subsample',random_state=42, n_jobs=-1),
    'SVM': SVC(kernel='linear', class_weight='balanced',probability=True, random_state=42),
}

In [55]:
res_w2v_p = train_embed_w2v('Parquet',         emb_w2v_p, df_parquet, modelos_w2v)
res_w2v_a = train_embed_w2v('ACCORD Official', emb_w2v_a, df_accord,  modelos_w2v)
res_w2v_b = train_embed_w2v('Base Construída', emb_w2v_b, df_base,    modelos_w2v)


Word2Vec | Base: Parquet

── Logistic Regression ──
              precision    recall  f1-score   support

   Não-Regra       0.80      0.68      0.73       342
       Regra       0.63      0.77      0.69       247

    accuracy                           0.71       589
   macro avg       0.71      0.72      0.71       589
weighted avg       0.73      0.71      0.71       589


── Random Forest ──
              precision    recall  f1-score   support

   Não-Regra       0.80      0.85      0.83       342
       Regra       0.78      0.70      0.74       247

    accuracy                           0.79       589
   macro avg       0.79      0.78      0.78       589
weighted avg       0.79      0.79      0.79       589


── SVM ──
              precision    recall  f1-score   support

   Não-Regra       0.80      0.68      0.73       342
       Regra       0.63      0.77      0.70       247

    accuracy                           0.72       589
   macro avg       0.72      0.72      0.72

In [56]:
df_summary_w2v = pd.DataFrame(res_w2v_p + res_w2v_a + res_w2v_b)
df_summary_w2v

,Base,Representação,Modelo,Accuracy,Precision,Recall,F1-Regra,F1-NRegra,F1-Macro,F1-Weighted
0,Parquet,Word2Vec,Logistic Regression,0.7131,0.6300,0.7652,0.6910,0.7322,0.7116,0.7149
1,Parquet,Word2Vec,Random Forest,0.7912,0.7768,0.7045,0.7389,0.8260,0.7824,0.7895
2,Parquet,Word2Vec,SVM,0.7165,0.6325,0.7733,0.6958,0.7345,0.7152,0.7183
3,ACCORD Official,Word2Vec,Logistic Regression,0.6767,0.5073,0.6753,0.5794,0.7374,0.6584,0.6853
4,ACCORD Official,Word2Vec,Random Forest,0.7666,0.6923,0.5260,0.5978,0.8356,0.7167,0.7572
5,ACCORD Official,Word2Vec,SVM,0.6745,0.5048,0.6883,0.5824,0.7333,0.6579,0.6836
6,Base Construída,Word2Vec,Logistic Regression,0.7419,0.6729,0.7669,0.7168,0.7629,0.7398,0.7432
7,Base Construída,Word2Vec,Random Forest,0.8069,0.7892,0.7458,0.7669,0.8351,0.8010,0.8061
8,Base Construída,Word2Vec,SVM,0.7401,0.6608,0.8008,0.7241,0.7543,0.7392,0.7414


In [57]:
print('\nMelhor por base (F1-Macro):')
for base in df_summary_w2v['Base'].unique():
    sub  = df_summary_w2v[df_summary_w2v['Base'] == base]
    best = sub.loc[sub['F1-Macro'].idxmax()]
    print(f'   {base:<20} → {best["Modelo"]:<22} F1-Macro={best["F1-Macro"]:.4f}')


Melhor por base (F1-Macro):
   Parquet              → Random Forest          F1-Macro=0.7824
   ACCORD Official      → Random Forest          F1-Macro=0.7167
   Base Construída      → Random Forest          F1-Macro=0.8010


In [58]:
df_summary_w2v.to_csv('../data/summary_word2vec.csv', index=False)

## Avaliando finetunning

In [59]:
# Verificar colab

## Avaliando Redes Neurais 

In [60]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, f1_score, roc_auc_score

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.BatchNorm1d(h),
                nn.Dropout(dropout)
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))   # saída binária
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

In [61]:
mlp_configs = [
    {
        'nome':        'MLP-Shallow',
        'hidden_dims': [256],              # 1 camada oculta
        'dropout':     0.3,
        'lr':          1e-3,
        'batch_size':  32,
        'epochs':      20,
    },
    {
        'nome':        'MLP-Medium',
        'hidden_dims': [512, 256],         # 2 camadas ocultas
        'dropout':     0.4,
        'lr':          1e-3,
        'batch_size':  32,
        'epochs':      20,
    },
    {
        'nome':        'MLP-Deep',
        'hidden_dims': [512, 256, 128],    # 3 camadas ocultas
        'dropout':     0.5,
        'lr':          5e-4,
        'batch_size':  64,
        'epochs':      30,
    },
]

In [62]:
def treinar_mlp(X_train, X_test, y_train, y_test, config, nome_base):
    device_mlp = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Converte para tensores
    X_tr = torch.FloatTensor(X_train).to(device_mlp)
    X_te = torch.FloatTensor(X_test).to(device_mlp)
    y_tr = torch.FloatTensor(y_train).to(device_mlp)
    y_te = torch.FloatTensor(y_test).to(device_mlp)

    # Peso para classe minoritária (desbalanceamento)
    pos_weight = torch.tensor(
        [(y_train == 0).sum() / (y_train == 1).sum()]
    ).to(device_mlp)

    # Dataset e DataLoader
    dataset_tr = TensorDataset(X_tr, y_tr)
    loader_tr  = DataLoader(dataset_tr,
                             batch_size=config['batch_size'],
                             shuffle=True)

    # Modelo
    model_mlp = MLP(
        input_dim   = X_train.shape[1],
        hidden_dims = config['hidden_dims'],
        dropout     = config['dropout']
    ).to(device_mlp)

    optimizer = torch.optim.Adam(model_mlp.parameters(), lr=config['lr'])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    # Treino
    best_f1, best_preds, best_probs = 0, None, None
    for epoch in range(config['epochs']):
        model_mlp.train()
        epoch_loss = 0
        for X_batch, y_batch in loader_tr:
            optimizer.zero_grad()
            logits = model_mlp(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        # Avaliação no teste
        model_mlp.eval()
        with torch.no_grad():
            logits_te = model_mlp(X_te)
            probs_te  = torch.sigmoid(logits_te).cpu().numpy()
            preds_te  = (probs_te >= 0.5).astype(int)

        f1 = f1_score(y_test, preds_te, average='macro', zero_division=0)
        scheduler.step(epoch_loss)

        if f1 > best_f1:
            best_f1    = f1
            best_preds = preds_te
            best_probs = probs_te

        if (epoch + 1) % 5 == 0:
            print(f'   Epoch {epoch+1:>2}/{config["epochs"]} | '
                  f'Loss={epoch_loss:.4f} | F1-Macro={f1:.4f}')

    return best_preds, best_probs

In [63]:
# ── Executa MLP nas três bases sobre embeddings RoBERTa (mean pooling) ────────
# Substitua emb_*_mp pelos embeddings que quiser testar
from sklearn.preprocessing import normalize

resultados_mlp = []

configs_bases = [
    ('Parquet',         X_train_tfidf_p, X_test_tfidf_p, y_train_p, y_test_p),
    ('ACCORD Official', X_train_tfidf_a, X_test_tfidf_a, y_train_a, y_test_a),
    ('Base Construída', X_train_tfidf_b, X_test_tfidf_b, y_train_b, y_test_b),
]

for nome_base, emb_train, emb_test, y_train, y_test in configs_bases:
    # Normaliza
    X_tr = normalize(emb_train, norm='l2')
    X_te = normalize(emb_test,  norm='l2')

    for cfg in mlp_configs:
        print(f'\n{"="*55}')
        print(f'MLP: {cfg["nome"]} | Base: {nome_base}')
        print(f'{"="*55}')

        preds, probs = treinar_mlp(X_tr, X_te, y_train, y_test,
                                    cfg, nome_base)

        print(classification_report(y_test, preds,
                                     target_names=['Não-Regra','Regra']))

        resultados_mlp.append({
            'Base':          nome_base,
            'Representação': 'RoBERTa + MLP',
            'Modelo':        cfg['nome'],
            'hidden_dims':   str(cfg['hidden_dims']),
            'dropout':       cfg['dropout'],
            'Accuracy':  round(f1_score(y_test, preds, average='weighted',  zero_division=0), 4),
            'F1-Regra':  round(f1_score(y_test, preds, pos_label=1,         zero_division=0), 4),
            'F1-NRegra': round(f1_score(y_test, preds, pos_label=0,         zero_division=0), 4),
            'F1-Macro':  round(f1_score(y_test, preds, average='macro',     zero_division=0), 4),
        })


MLP: MLP-Shallow | Base: Parquet


TypeError: sparse array length is ambiguous; use getnnz() or shape[0]

In [ ]:
df_summary_mlp = pd.DataFrame(resultados_mlp)
df_summary_mlp

In [ ]:
df_summary_mlp.to_csv('../data/summary_mlp.csv', index=False)